In [14]:
import pandas as pd
df_politicas = pd.read_csv(r"C:\....\BaseBNuevoCruce.csv" , encoding='latin-1', sep = ";")
df_politicas.head(1)

,0,Código DANE,Entidad territorial,Nombre del plan,Código de indicador de producto (SisPT),Línea estratégica,Código del sector,Sector (MGA),Código del programa (MGA),Programa (MGA),Código del producto (MGA),Producto (MGA),Código de indicador de producto (MGA),Indicador de Producto(MGA),Personalización de Indicador de Producto
0,5_Antioquia_POR ANTIOQUIA FIRME 2024-2027_LE-3...,5,Antioquia,POR ANTIOQUIA FIRME 2024-2027,IP-184,LE-3-Inversión desde la confianza,17,Agricultura y desarrollo rural\n,1702,Inclusión productiva de pequeños productores r...,1702021,Servicio de acompañamiento productivo y empres...,170202100,Unidades productivas beneficiadas,53020201-Unidades productivas agropecuarias be...


In [15]:
import pandas as pd
df_proyectos = pd.read_csv(r"C:.........\BaseANuevoCruce.csv" , encoding='latin-1', sep = ";")
df_proyectos['codigo_proyecto'] = df_proyectos.index
df_proyectos.head(1)

,Indicadores de producto PATR,Indicadores de producto Plan indicativo,codigo_proyecto
0,Actividades culturales para la promoción de la...,Actividades culturales para la promoción de la...,0


# Modelo 1

In [21]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

def top10_politicas_por_proyecto(
    df_politicas: pd.DataFrame,
    df_proyectos: pd.DataFrame,
    col_text_politica: str,
    col_text_proyecto: str,
    col_id_proyecto: str,
    #codigo_indicador: str,
    #indicador_producto: str,
    #medicion_indicador: str,
    #unidad_indicador: str,
    top_k: int = 10,
    min_score: float = 0.35,
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    batch_size: int = 64,
    device: str = "cpu",  # 👈 recomendado: evita problemas CUDA
) -> pd.DataFrame:
    """
    Devuelve TOP-K políticas por cada proyecto, usando similitud coseno sobre embeddings.
    Salida en formato 'long' (top_k filas por proyecto, o menos si no pasan min_score).
    """

    # 0) Validaciones mínimas
    for c in [col_text_politica]: #, codigo_indicador, indicador_producto, medicion_indicador, unidad_indicador]:
        if c not in df_politicas.columns:
            raise ValueError(f"df_politicas no tiene la columna requerida: {c}")
    for c in [col_text_proyecto, col_id_proyecto]:
        if c not in df_proyectos.columns:
            raise ValueError(f"df_proyectos no tiene la columna requerida: {c}")

    # 1) Modelo (control explícito del device)
    model = SentenceTransformer(model_name, device=device)

    # 2) Textos
    pol_texts = df_politicas[col_text_politica].fillna("").astype(str).tolist()
    proy_texts = df_proyectos[col_text_proyecto].fillna("").astype(str).tolist()

    # 3) Embeddings normalizados (coseno = dot)
    E_pol = model.encode(pol_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    E_proy = model.encode(proy_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    # 4) Vecinos: para cada PROYECTO buscamos POLÍTICAS cercanas
    nn = NearestNeighbors(n_neighbors=top_k, metric="cosine", algorithm="brute")
    nn.fit(E_pol)  # 👈 entrenamos con políticas

    distances, indices = nn.kneighbors(E_proy, return_distance=True)  # 👈 consultamos con proyectos
    scores = 1.0 - distances

    # 5) Construcción del resultado (top_k por proyecto)
    out_rows = []
    for i in range(len(df_proyectos)):
        base = df_proyectos.iloc[i].to_dict()

        for rank in range(top_k):
            j = int(indices[i, rank])  # índice de política
            sc = float(scores[i, rank])

            if sc < min_score:
                continue

            out_rows.append({
                **base,
                #"matched_politica_id": df_politicas.iloc[j][codigo_indicador],
                "matched_politica_text": df_politicas.iloc[j][col_text_politica],
                #"matched_indicador_producto": df_politicas.iloc[j][indicador_producto],
                #"matched_indicador_medicion": df_politicas.iloc[j][medicion_indicador],
                #"matched_indicador_unidad": df_politicas.iloc[j][unidad_indicador],
                "similarity_score": sc,
                "rank": rank + 1
            })

    df_out = pd.DataFrame(out_rows)

    # Orden (útil para inspección)
    df_out = df_out.sort_values([col_id_proyecto, "rank"], ascending=[True, True]).reset_index(drop=True)

    return df_out


# =========================
# EJEMPLO DE USO (ajusta tus nombres)
# =========================
df_matches = top10_politicas_por_proyecto(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica= "Indicador de Producto(MGA)",  #"Producto_enriquecido_2",
    col_text_proyecto= "Indicadores de producto PATR",  #"id_proyecto",
    col_id_proyecto="codigo_proyecto",
    #codigo_indicador="codigo_indicador",
    #indicador_producto="indicador_producto",
    #medicion_indicador="medicion_indicador",
    #unidad_indicador="unidad_indicador",
    top_k=10,
    min_score=0.35,
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    batch_size=64,
    device="cpu"   # si quieres intentar GPU: "cuda" pero con batch_size=8
)

# Exportar a Excel
ruta = r"C:\TrabajoAndrea\nuevo_cruce_top10.xlsx"
df_matches.to_excel(ruta, index=False)
print("Guardado:", ruta)


C:\Users\andre\anaconda3\envs\matchcpu\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|████████████████████████████████████████████████████████

Guardado: C:\TrabajoAndrea\nuevo_cruce_top10.xlsx


# Modelo 2

In [23]:
import re
import numpy as np
import pandas as pd
from typing import Optional, List, Dict, Any

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.neighbors import NearestNeighbors


# -----------------------------
# 1) Limpieza ligera (no agresiva)
# -----------------------------
def _clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s


# -----------------------------
# 2) Formateo por familia de modelo (E5 / BGE)
# -----------------------------
def _format_for_model(texts: List[str], mode: str, model_name: str) -> List[str]:
    """
    Para modelos tipo E5/BGE es MUY importante usar prefijos:
      - query: ...   (cuando es la política / consulta)
      - passage: ... (cuando es el proyecto / documento)
    """
    model_lower = model_name.lower()

    if ("e5" in model_lower) or ("bge" in model_lower):
        prefix = "query: " if mode == "query" else "passage: "
        return [prefix + t for t in texts]

    # Para otros modelos, devolver tal cual
    return texts


# -----------------------------
# 3) Función principal mejorada
# -----------------------------
def semantic_project_merge_advanced(
    df_politicas: pd.DataFrame,
    df_proyectos: pd.DataFrame,
    col_text_politica: str,
    col_text_proyecto: str,
    col_id_proyecto: str,
    #codigo_indicador: str,
    #indicador_producto: str,
    #medicion_indicador: str,
    #unidad_indicador: str,
    top_k: int = 3,
    min_bi_score: float = 0.45,
    top_n_candidates: int = 50,
    # Embeddings (elige UNO)
    model_name: str = "BAAI/bge-m3",  # alternativa muy buena: "intfloat/multilingual-e5-large"
    batch_size: int = 256,
    # Re-ranking (cross-encoder)
    use_rerank: bool = True,
    cross_encoder_name: str = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    rerank_keep_top_k: Optional[int] = None,  # si None, usa top_k
) -> pd.DataFrame:
    """
    Para cada PROYECTO, encuentra las POLÍTICAS más similares:
      1) bi-encoder embeddings (coseno) para recuperar top_n_candidates
      2) cross-encoder para re-rank (mejora mucho la precisión)

    Retorna formato 'long' con hasta top_k matches por proyecto.

    NOTAS IMPORTANTES:
    - min_bi_score: umbral del coseno en la etapa bi-encoder (bájalo si se están quedando sin matches)
    - top_n_candidates: candidatos a re-rankear (20-100 suele ir bien)
    """

    # -----------------------------
    # Validaciones básicas
    # -----------------------------
    needed_pol = [col_text_politica]  #, codigo_indicador, indicador_producto, medicion_indicador, unidad_indicador]
    needed_proy = [col_text_proyecto, col_id_proyecto]
    for c in needed_pol:
        if c not in df_politicas.columns:
            raise ValueError(f"df_politicas no tiene la columna: {c}")
    for c in needed_proy:
        if c not in df_proyectos.columns:
            raise ValueError(f"df_proyectos no tiene la columna: {c}")

    if rerank_keep_top_k is None:
        rerank_keep_top_k = top_k

    # -----------------------------
    # 1) Textos limpios
    # -----------------------------
    pol_texts_raw = df_politicas[col_text_politica].fillna("").astype(str).map(_clean_text).tolist()
    proy_texts_raw = df_proyectos[col_text_proyecto].fillna("").astype(str).map(_clean_text).tolist()

    # -----------------------------
    # 2) Embeddings (bi-encoder)
    # -----------------------------
    bi = SentenceTransformer(model_name, device="cpu")

    # Formateo por modelo (E5/BGE)
    pol_texts = _format_for_model(pol_texts_raw, mode="query", model_name=model_name)
    proy_texts = _format_for_model(proy_texts_raw, mode="passage", model_name=model_name)

    E_pol = bi.encode(pol_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    E_proy = bi.encode(proy_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    # -----------------------------
    # 3) Recuperación top-N por coseno (NearestNeighbors con distancia coseno)
    #    Como quieres: PROYECTO -> POLÍTICAS
    # -----------------------------
    top_n_candidates = max(top_k, int(top_n_candidates))
    nn = NearestNeighbors(n_neighbors=top_n_candidates, metric="cosine", algorithm="brute")
    nn.fit(E_pol)

    distances, indices = nn.kneighbors(E_proy, return_distance=True)
    bi_scores = 1.0 - distances  # coseno

    # -----------------------------
    # 4) Re-ranking Cross-Encoder (opcional)
    # -----------------------------
    ce = CrossEncoder(cross_encoder_name) if use_rerank else None

    out_rows: List[Dict[str, Any]] = []

    for i in range(len(df_proyectos)):
        base = df_proyectos.iloc[i].to_dict()

        # Candidatos por bi-encoder con filtro
        cand_pairs = []
        cand_meta = []
        for r in range(top_n_candidates):
            j = int(indices[i, r])
            sc_bi = float(bi_scores[i, r])
            if sc_bi < float(min_bi_score):
                continue
            cand_pairs.append([pol_texts_raw[j], proy_texts_raw[i]])  # (política, proyecto)
            cand_meta.append((j, sc_bi))

        if not cand_meta:
            continue

        # Si hay rerank, recalcular orden usando cross-encoder
        if use_rerank:
            # Cross-encoder típicamente espera pares (texto1, texto2)
            # Aquí: (política, proyecto)
            ce_scores = ce.predict(cand_pairs)  # array de relevancia (no necesariamente 0-1)
            order = np.argsort(-ce_scores)  # descendente
        else:
            ce_scores = np.array([np.nan] * len(cand_meta))
            order = np.argsort([-m[1] for m in cand_meta])  # descendente por bi-score

        kept = 0
        for idx_ord in order:
            j, sc_bi = cand_meta[int(idx_ord)]
            sc_ce = float(ce_scores[int(idx_ord)]) if use_rerank else np.nan

            out_rows.append({
                **base,
                #"matched_politica_id": df_politicas.iloc[j][codigo_indicador],
                "matched_politica_text": df_politicas.iloc[j][col_text_politica],
                #"matched_indicador_producto": df_politicas.iloc[j][indicador_producto],
                #"matched_indicador_medicion": df_politicas.iloc[j][medicion_indicador],
                #"matched_indicador_unidad": df_politicas.iloc[j][unidad_indicador],
                "bi_similarity_score": sc_bi,         # coseno (0-1)
                "rerank_score": sc_ce,                # score cross-encoder (escala del modelo)
                "rank": kept + 1
            })

            kept += 1
            if kept >= rerank_keep_top_k:
                break

    return pd.DataFrame(out_rows)

df_match = semantic_project_merge_advanced(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica= "Indicador de Producto(MGA)",  #"Producto_enriquecido_2",
    col_text_proyecto= "Indicadores de producto PATR",  #"id_proyecto",
    col_id_proyecto="codigo_proyecto",
    #codigo_indicador = "codigo_indicador",
    #indicador_producto = "indicador_producto",
    #medicion_indicador = "medicion_indicador",
    #unidad_indicador= "unidad_indicador",
    top_k=4,
    top_n_candidates=80,
    min_bi_score=0.40,
    model_name="BAAI/bge-m3",  # o "intfloat/multilingual-e5-large"
    use_rerank=True,
    cross_encoder_name="cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)


In [29]:
df_match = semantic_project_merge_advanced(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica= "Indicador de Producto(MGA)",  #"Producto_enriquecido_2",
    col_text_proyecto= "Indicadores de producto PATR",  #"id_proyecto",
    col_id_proyecto="codigo_proyecto",
    #codigo_indicador = "codigo_indicador",
    #indicador_producto = "indicador_producto",
    #medicion_indicador = "medicion_indicador",
    #unidad_indicador= "unidad_indicador",
    top_k=4,
    top_n_candidates=80,
    min_bi_score=0.40,
    model_name="BAAI/bge-m3",  # o "intfloat/multilingual-e5-large"
    use_rerank=True,
    cross_encoder_name="cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 2/2 [00:57<00:00, 28.67s/it]


In [30]:
df_match.to_csv(r"C:\TrabajoAndrea\nuevo_cruce_top10_Modelo2.csv")

# Modelo 3

In [18]:
#pip install -U pandas numpy scikit-learn sentence-transformers requests
import requests
print(requests.get("http://localhost:11434/api/tags", timeout=10).status_code)


200


In [20]:
# ============================================================
# MATCHING PRECISO: PROYECTO -> TOP-10 POLÍTICAS
# (Embeddings BGE-M3 + Rerank LLM (Ollama) + Fallback robusto)
# ============================================================

import re
import json
import time
import numpy as np
import pandas as pd
import requests
from typing import List, Dict, Any, Optional, Tuple

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


# =========================
# 0) FIX SSL (si HF da CERTIFICATE_VERIFY_FAILED)
# =========================
def enable_hf_ssl_fix() -> str:
    """
    Si te da SSLCertVerificationError al bajar modelos de HuggingFace,
    ejecuta esto ANTES de crear SentenceTransformer.
    """
    import os, certifi
    os.environ["SSL_CERT_FILE"] = certifi.where()
    os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
    return certifi.where()


# =========================
# 1) Utilidades
# =========================
def _clean_text(s: Any) -> str:
    s = "" if s is None or (isinstance(s, float) and np.isnan(s)) else str(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s


def _format_for_model(texts: List[str], mode: str, model_name: str) -> List[str]:
    """
    Para modelos tipo E5/BGE, usar prefijos mejora la calidad:
      - query: (consulta) -> política
      - passage: (documento) -> proyecto
    """
    ml = model_name.lower()
    if ("e5" in ml) or ("bge" in ml):
        prefix = "query: " if mode == "query" else "passage: "
        return [prefix + t for t in texts]
    return texts


def _extract_json_loose(text: str) -> Dict[str, Any]:
    """
    Extrae JSON aunque el modelo meta texto extra.
    """
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No se encontró un bloque JSON en la respuesta del LLM.")
    return json.loads(m.group(0).strip())


def _validate_llm_schema(obj: Dict[str, Any]) -> None:
    if "selections" not in obj or not isinstance(obj["selections"], list):
        raise ValueError("LLM: falta 'selections' o no es lista.")
    if len(obj["selections"]) == 0:
        raise ValueError("LLM: 'selections' viene vacía.")
    # Validación ligera por item
    for it in obj["selections"]:
        if not isinstance(it, dict):
            raise ValueError("LLM: selection no es dict.")
        for k in ["candidate_index", "score", "reason"]:
            if k not in it:
                raise ValueError(f"LLM: falta campo {k}.")
        sc = float(it["score"])
        if sc < 0 or sc > 1:
            raise ValueError("LLM: score fuera de [0,1].")


def ollama_is_up(url: str = "http://localhost:11434/api/tags", timeout: int = 5) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False


# =========================
# 2) Cliente LLM (Ollama)
# =========================
def ollama_rerank(
    politica_candidates: List[Tuple[int, str, float]],
    project_text: str,
    top_k_llm: int,
    ollama_url: str = "http://localhost:11434/api/generate",
    model_name: str = "deepseek-r1:7b",
    temperature: float = 0.0,
    timeout_sec: int = 600,
    max_retries: int = 3
) -> Dict[str, Any]:
    """
    politica_candidates: lista de (policy_index_en_df, policy_text, bi_score)
    Devuelve JSON:
      { "selections": [ {"candidate_index": i, "score": 0-1, "reason": "..."} ] }
    candidate_index se refiere al índice dentro de politica_candidates
    """

    candidates_txt = "\n".join([
        f"{i}. (bi_score={bi_score:.3f}) {pol_txt}"
        for i, (_, pol_txt, bi_score) in enumerate(politica_candidates)
    ])

    prompt = f"""
Eres un experto en políticas públicas y formulación de proyectos.

TAREA:
Selecciona las {top_k_llm} POLÍTICAS que mejor correspondan al PROYECTO.

PROYECTO:
\"\"\"{project_text}\"\"\"

POLÍTICAS CANDIDATAS:
{candidates_txt}

CRITERIOS (prioridad):
1) Misma finalidad/objetivo (qué problema público atiende)
2) Misma población objetivo / beneficiarios
3) Misma intervención o instrumento (infraestructura, subsidio, regulación, fortalecimiento institucional, tecnología, etc.)
4) Mismo sector/tema (salud, educación, agua, vivienda, empleo, ambiente, seguridad, etc.)
5) Nivel de especificidad: prefiere la política más específica y directamente aplicable

SALIDA:
Devuelve SOLO JSON (sin texto adicional) con el formato:

{{
  "selections": [
    {{"candidate_index": 0, "score": 0.0, "reason": "1 frase"}}
  ]
}}

REGLAS:
- score entre 0 y 1.
- reason máximo 1 frase.
- candidate_index debe ser un índice del listado de candidatas.
""".strip()

    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }

    last_err: Optional[Exception] = None
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(ollama_url, json=payload, timeout=timeout_sec)
            if r.status_code != 200:
                raise RuntimeError(f"Ollama HTTP {r.status_code}: {r.text[:300]}")

            obj = _extract_json_loose(r.json().get("response", ""))
            _validate_llm_schema(obj)
            return obj

        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 10))

    raise RuntimeError(f"LLM rerank falló tras {max_retries} intentos. Último error: {last_err}")


# =========================
# 3) Función principal (Precisión)
# =========================
def semantic_project_merge_llm_rerank_preciso(
    df_politicas: pd.DataFrame,
    df_proyectos: pd.DataFrame,
    col_text_politica: str,
    col_text_proyecto: str,
    col_id_proyecto: str,
    #codigo_indicador: str,
    #indicador_producto: str,
    #medicion_indicador: str,
    #unidad_indicador: str,

    # Salida
    top_k: int = 10,                 # TOP 10 final por proyecto
    top_k_llm: int = 5,              # TOP 5 elegidos por LLM (más preciso/estable)

    # Embeddings (recall alto)
    model_name: str = "BAAI/bge-m3",
    batch_size: int = 8,             # RTX 2050 4GB: 8 o 16 (si OOM, baja)
    top_n_candidates: int = 400,     # alto para recall
    min_bi_score: float = 0.25,      # no recortar demasiado

    # LLM rerank (precisión)
    use_llm_rerank: bool = True,
    llm_candidates_cap: int = 60,    # candidatos que ve el LLM (40-80)
    ollama_url: str = "http://localhost:11434/api/generate",
    llm_model_name: str = "deepseek-r1:7b",
    llm_temperature: float = 0.0,
    llm_timeout_sec: int = 600,

    # Score combinado final
    weight_llm: float = 0.65,
    weight_bi: float = 0.35,

    sleep_between_projects_sec: float = 0.0
) -> pd.DataFrame:
    """
    Por cada PROYECTO:
      1) Recupera top_n_candidates POLÍTICAS por embeddings (coseno)
      2) (Opcional) LLM re-rank elige top_k_llm
      3) Completa a top_k con embeddings
      4) Calcula final_score = w_llm*llm_score + w_bi*bi_score (si no hay llm_score, usa bi_score)
    """

    # --- Validaciones ---
    needed_pol = [col_text_politica], #, codigo_indicador, indicador_producto, medicion_indicador, unidad_indicador]
    needed_proy = [col_text_proyecto, col_id_proyecto]
    #for c in needed_pol:
        #if c not in df_politicas.columns:
            #raise ValueError(f"df_politicas no tiene columna requerida: {c}")
    #for c in needed_proy:
        #if c not in df_proyectos.columns:
            #raise ValueError(f"df_proyectos no tiene columna requerida: {c}")

    top_k = int(top_k)
    top_k_llm = int(min(top_k_llm, top_k))

    if llm_candidates_cap < top_k_llm:
        llm_candidates_cap = top_k_llm

    # Si Ollama no está arriba, apagamos LLM automáticamente
    if use_llm_rerank and not ollama_is_up("http://localhost:11434/api/tags", timeout=5):
        print("⚠️ Ollama no está accesible. Continuo solo con embeddings (sin LLM).")
        use_llm_rerank = False

    # --- Textos limpios ---
    pol_texts_raw = df_politicas[col_text_politica].fillna("").astype(str).map(_clean_text).tolist()
    proy_texts_raw = df_proyectos[col_text_proyecto].fillna("").astype(str).map(_clean_text).tolist()

    # --- Device (cuda si disponible) ---
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"

    # --- Embeddings ---
    bi = SentenceTransformer(model_name, device=device)

    pol_texts_for_emb = _format_for_model(pol_texts_raw, mode="query", model_name=model_name)
    proy_texts_for_emb = _format_for_model(proy_texts_raw, mode="passage", model_name=model_name)

    E_pol = bi.encode(pol_texts_for_emb, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    E_proy = bi.encode(proy_texts_for_emb, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    # --- Recuperación top-N ---
    top_n_candidates = max(top_k, int(top_n_candidates))
    nn = NearestNeighbors(n_neighbors=top_n_candidates, metric="cosine", algorithm="brute")
    nn.fit(E_pol)

    distances, indices = nn.kneighbors(E_proy, return_distance=True)
    bi_scores = 1.0 - distances

    out_rows: List[Dict[str, Any]] = []

    for i in range(len(df_proyectos)):
        base = df_proyectos.iloc[i].to_dict()
        project_text = proy_texts_raw[i]

        # 1) candidatos por embeddings
        candidates: List[Tuple[int, str, float]] = []
        for r in range(top_n_candidates):
            j = int(indices[i, r])
            sc = float(bi_scores[i, r])
            if sc < float(min_bi_score):
                continue
            candidates.append((j, pol_texts_raw[j], sc))

        if not candidates:
            continue

        candidates_for_llm = candidates[: min(len(candidates), int(llm_candidates_cap))]

        # 2) LLM rerank robusto (si falla, fallback a embeddings)
        llm_selections: List[Dict[str, Any]] = []
        llm_failed_reason: Optional[str] = None

        if use_llm_rerank:
            try:
                llm_json = ollama_rerank(
                    politica_candidates=candidates_for_llm,
                    project_text=project_text,
                    top_k_llm=top_k_llm,
                    ollama_url=ollama_url,
                    model_name=llm_model_name,
                    temperature=llm_temperature,
                    timeout_sec=llm_timeout_sec
                )
                llm_selections = llm_json.get("selections", []) or []
            except Exception as e:
                llm_failed_reason = f"{type(e).__name__}: {str(e)[:200]}"
                llm_selections = []

        # 3) Construcción: primero LLM (top_k_llm), luego completar a top_k por embeddings
        used_policy_idx = set()
        rank = 0

        # 3a) Aplicar selecciones LLM
        if llm_selections:
            for sel in llm_selections:
                try:
                    cand_idx = int(sel["candidate_index"])
                    llm_score = float(sel["score"])
                    llm_reason = str(sel["reason"])
                except Exception:
                    continue

                if cand_idx < 0 or cand_idx >= len(candidates_for_llm):
                    continue

                pol_j, _, bi_sc = candidates_for_llm[cand_idx]
                if pol_j in used_policy_idx:
                    continue

                used_policy_idx.add(pol_j)
                rank += 1

                # score final
                final_score = weight_llm * llm_score + weight_bi * float(bi_sc)

                out_rows.append({
                    **base,
                    #"matched_politica_id": df_politicas.iloc[pol_j][codigo_indicador],
                    "matched_politica_text": df_politicas.iloc[pol_j][col_text_politica],
                    #"matched_indicador_producto": df_politicas.iloc[pol_j][indicador_producto],
                    #"matched_indicador_medicion": df_politicas.iloc[pol_j][medicion_indicador],
                    #"matched_indicador_unidad": df_politicas.iloc[pol_j][unidad_indicador],
                    "bi_similarity_score": float(bi_sc),
                    "llm_score": float(llm_score),
                    "llm_reason": llm_reason,
                    "final_score": float(final_score),
                    "rank": rank,
                    "device_used": device,
                    "llm_failed": False
                })

                if rank >= top_k:
                    break

        # 3b) Completar con embeddings hasta top_k
        if rank < top_k:
            fallback = sorted(candidates, key=lambda x: x[2], reverse=True)
            for (pol_j, _, bi_sc) in fallback:
                if rank >= top_k:
                    break
                if pol_j in used_policy_idx:
                    continue

                used_policy_idx.add(pol_j)
                rank += 1

                # sin llm_score: final_score=bi_score
                out_rows.append({
                    **base,
                    #"matched_politica_id": df_politicas.iloc[pol_j][codigo_indicador],
                    "matched_politica_text": df_politicas.iloc[pol_j][col_text_politica],
                    #"matched_indicador_producto": df_politicas.iloc[pol_j][indicador_producto],
                    #"matched_indicador_medicion": df_politicas.iloc[pol_j][medicion_indicador],
                    #"matched_indicador_unidad": df_politicas.iloc[pol_j][unidad_indicador],
                    "bi_similarity_score": float(bi_sc),
                    "llm_score": np.nan,
                    "llm_reason": "fallback_by_embeddings" if llm_failed_reason is None else f"fallback_by_embeddings (llm_failed: {llm_failed_reason})",
                    "final_score": float(bi_sc),
                    "rank": rank,
                    "device_used": device,
                    "llm_failed": llm_failed_reason is not None
                })

        time.sleep(sleep_between_projects_sec)

    df_out = pd.DataFrame(out_rows)

    # Orden final: por proyecto + rank (ya viene), y opcionalmente por final_score desc
    if not df_out.empty and col_id_proyecto in df_out.columns:
        df_out = df_out.sort_values([col_id_proyecto, "rank"], ascending=[True, True]).reset_index(drop=True)

    return df_out


# =========================
# 4) Exportar a Excel
# =========================
def export_matches_to_excel(
    df_out: pd.DataFrame,
    path_xlsx: str,
    project_id_col: str,
    make_top1_sheet: bool = True
) -> None:
    """
    Exporta a Excel:
      - Hoja 1: top_k_por_proyecto (completo)
      - Hoja 2: top_1_por_proyecto (opcional)
    """
    # Requiere openpyxl o xlsxwriter
    with pd.ExcelWriter(path_xlsx, engine="xlsxwriter") as writer:
        df_out.to_excel(writer, sheet_name="top_k_por_proyecto", index=False)

        if make_top1_sheet and (project_id_col in df_out.columns):
            df_top1 = (
                df_out
                .sort_values([project_id_col, "rank"])
                .groupby(project_id_col, as_index=False)
                .first()
            )
            df_top1.to_excel(writer, sheet_name="top_1_por_proyecto", index=False)


In [28]:
df_out = semantic_project_merge_llm_rerank_preciso(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica="Indicador de Producto(MGA)",
    col_text_proyecto="Indicadores de producto PATR",
    col_id_proyecto="codigo_proyecto",

    top_k=10,
    top_k_llm=5,

    model_name="BAAI/bge-m3",
    batch_size=16,
    top_n_candidates=120,
    min_bi_score=0.25,


    use_llm_rerank=True,
    llm_model_name="qwen2.5:3b-instruct",
    llm_candidates_cap=15,
    llm_timeout_sec=180,
    llm_temperature=0.0
)


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:41<00:00,  2.20s/it]


In [29]:
ruta = r"C:\TrabajoAndrea\nuevo_cruce_modelo5.xlsx"

export_matches_to_excel(
    df_out=df_out,
    path_xlsx=ruta,
    project_id_col="codigo_proyecto",
    make_top1_sheet=True
)

print("Excel guardado en:", ruta)


Excel guardado en: C:\TrabajoAndrea\nuevo_cruce_modelo5.xlsx


# Modelo 4

In [61]:
import os
import re
import json
import time
import hashlib
import numpy as np
import pandas as pd
import requests
from typing import List, Dict, Any, Optional, Tuple

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


# =========================
# (Opcional) Fix SSL HuggingFace
# =========================
def enable_hf_ssl_fix() -> str:
    import certifi
    os.environ["SSL_CERT_FILE"] = certifi.where()
    os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
    return certifi.where()


# =========================
# Utilidades
# =========================
def _clean_text(s: Any) -> str:
    s = "" if s is None or (isinstance(s, float) and np.isnan(s)) else str(s)
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _format_for_model(texts: List[str], mode: str, model_name: str) -> List[str]:
    ml = model_name.lower()
    if ("e5" in ml) or ("bge" in ml):
        prefix = "query: " if mode == "query" else "passage: "
        return [prefix + t for t in texts]
    return texts

def ollama_is_up(url: str = "http://localhost:11434/api/tags", timeout: int = 3) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code == 200
    except Exception:
        return False

def _extract_json_loose(text: str) -> Dict[str, Any]:
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No se encontró JSON en respuesta del LLM.")
    return json.loads(m.group(0).strip())

def _validate_llm_schema(obj: Dict[str, Any]) -> None:
    if "selections" not in obj or not isinstance(obj["selections"], list) or len(obj["selections"]) == 0:
        raise ValueError("LLM: selections inválido/vacío.")
    for it in obj["selections"]:
        for k in ["candidate_index", "score", "reason"]:
            if k not in it:
                raise ValueError(f"LLM: falta {k}.")
        sc = float(it["score"])
        if sc < 0 or sc > 1:
            raise ValueError("LLM: score fuera de [0,1].")


# =========================
# Cache de rerank (en disco)
# =========================
def _cache_key(project_text: str, candidate_policy_ids: List[Any], model_name: str, top_k_llm: int) -> str:
    h = hashlib.sha256()
    h.update(model_name.encode("utf-8"))
    h.update(str(top_k_llm).encode("utf-8"))
    h.update(project_text.encode("utf-8"))
    h.update(("|".join(map(str, candidate_policy_ids))).encode("utf-8"))
    return h.hexdigest()

def _cache_load(cache_dir: str, key: str) -> Optional[Dict[str, Any]]:
    path = os.path.join(cache_dir, f"{key}.json")
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return None
    return None

def _cache_save(cache_dir: str, key: str, obj: Dict[str, Any]) -> None:
    os.makedirs(cache_dir, exist_ok=True)
    path = os.path.join(cache_dir, f"{key}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)


# =========================
# LLM rerank (Ollama)
# =========================
def ollama_rerank(
    politica_candidates: List[Tuple[int, str, float]],
    project_text: str,
    top_k_llm: int,
    ollama_url: str,
    model_name: str,
    temperature: float,
    timeout_sec: int,
    max_retries: int = 2
) -> Dict[str, Any]:
    # Prompt compacto: manda menos texto
    # (bi_score + política resumida)
    candidates_txt = "\n".join([
        f"{i}. (bi={bi_score:.3f}) {pol_txt}"
        for i, (_, pol_txt, bi_score) in enumerate(politica_candidates)
    ])

    prompt = f"""
Selecciona las {top_k_llm} políticas más alineadas con el proyecto.
Criterios: mismo objetivo, población, instrumento y sector. Prefiere la más específica.

PROYECTO:
{project_text}

POLÍTICAS CANDIDATAS:
{candidates_txt}

Devuelve SOLO JSON:
{{
  "selections": [
    {{"candidate_index": 0, "score": 0.0, "reason": "1 frase"}}
  ]
}}
""".strip()

    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature}
    }

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(ollama_url, json=payload, timeout=timeout_sec)
            if r.status_code != 200:
                raise RuntimeError(f"Ollama HTTP {r.status_code}: {r.text[:200]}")
            obj = _extract_json_loose(r.json().get("response", ""))
            _validate_llm_schema(obj)
            return obj
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 6))

    raise RuntimeError(f"LLM rerank falló. Último error: {last_err}")


# =========================
# Matching optimizado (LLM solo en casos difíciles)
# =========================
def match_proyecto_to_politicas_optimizado(
    df_politicas: pd.DataFrame,
    df_proyectos: pd.DataFrame,
    col_text_politica: str,
    col_text_proyecto: str,
    col_id_proyecto: str,
    #codigo_indicador: str,
    #indicador_producto: str,
    #medicion_indicador: str,
    #unidad_indicador: str,

    # salida
    top_k: int = 10,

    # embeddings
    embed_model: str = "BAAI/bge-m3",
    batch_size: int = 16,
    top_n_candidates: int = 200,    # suficiente + rápido
    min_bi_score: float = 0.25,

    # gating (CUÁNDO llamar LLM)
    # Si embeddings están "seguros", NO se llama LLM.
    confident_top1_threshold: float = 0.42,  # si top1 >= esto...
    confident_margin_threshold: float = 0.06, # ...y (top1-top2) >= esto => aceptar embeddings

    # LLM (solo si difícil)
    use_llm_rerank: bool = True,
    ollama_url: str = "http://localhost:11434/api/generate",
    llm_model: str = "deepseek-r1:7b",
    llm_timeout_sec: int = 240,     # baja a 240 para no colgar; fallback se activa
    llm_temperature: float = 0.0,
    llm_candidates_cap: int = 40,   # MUY importante para velocidad
    top_k_llm: int = 5,             # el LLM elige 5 (más preciso), completas a 10 con embeddings

    # cache
    cache_dir: str = "./ollama_rerank_cache",

    # score final
    w_llm: float = 0.65,
    w_bi: float = 0.35,

    # logging
    verbose_every: int = 200
) -> pd.DataFrame:

    # Validaciones
    for c in [col_text_politica]: #, codigo_indicador, indicador_producto, medicion_indicador, unidad_indicador]:
        if c not in df_politicas.columns:
            raise ValueError(f"df_politicas no tiene columna: {c}")
    for c in [col_text_proyecto, col_id_proyecto]:
        if c not in df_proyectos.columns:
            raise ValueError(f"df_proyectos no tiene columna: {c}")

    if use_llm_rerank and not ollama_is_up():
        print("⚠️ Ollama no accesible. Continuo SOLO con embeddings.")
        use_llm_rerank = False

    # Textos
    pol_texts_raw = df_politicas[col_text_politica].fillna("").astype(str).map(_clean_text).tolist()
    proy_texts_raw = df_proyectos[col_text_proyecto].fillna("").astype(str).map(_clean_text).tolist()

    # Device
    try:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"

    # Embeddings
    bi = SentenceTransformer(embed_model, device=device)

    pol_for_emb = _format_for_model(pol_texts_raw, "query", embed_model)
    proy_for_emb = _format_for_model(proy_texts_raw, "passage", embed_model)

    E_pol = bi.encode(pol_for_emb, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    E_proy = bi.encode(proy_for_emb, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)

    # NN
    top_n_candidates = max(top_k, int(top_n_candidates))
    nn = NearestNeighbors(n_neighbors=top_n_candidates, metric="cosine", algorithm="brute")
    nn.fit(E_pol)

    distances, indices = nn.kneighbors(E_proy, return_distance=True)
    bi_scores = 1.0 - distances

    out_rows: List[Dict[str, Any]] = []

    llm_calls = 0
    llm_used = 0

    for i in range(len(df_proyectos)):
        if verbose_every and (i % verbose_every == 0) and i > 0:
            print(f"[{i}/{len(df_proyectos)}] LLM calls={llm_calls}, LLM used={llm_used}")

        base = df_proyectos.iloc[i].to_dict()
        project_text = proy_texts_raw[i]

        # candidatos por embeddings
        candidates = []
        for r in range(top_n_candidates):
            j = int(indices[i, r])
            sc = float(bi_scores[i, r])
            if sc < min_bi_score:
                continue
            candidates.append((j, pol_texts_raw[j], sc))

        if not candidates:
            continue

        # Gating: decidir si usar LLM
        # top1 y top2 (si existe)
        top1 = candidates[0][2]
        top2 = candidates[1][2] if len(candidates) > 1 else 0.0
        margin = float(top1 - top2)

        is_confident = (top1 >= confident_top1_threshold) and (margin >= confident_margin_threshold)

        # Por defecto: no LLM (rápido)
        selections = []
        llm_failed = False
        llm_reason = ""

        # Si NO confiable => LLM
        if use_llm_rerank and (not is_confident):
            llm_used += 1

            # cap candidatos al LLM
            candidates_for_llm = candidates[: min(len(candidates), llm_candidates_cap)]

            # cache key por ids de política (usa codigo_indicador como id estable)
            candidate_ids = [df_politicas.iloc[j][codigo_indicador] for (j, _, _) in candidates_for_llm]
            key = _cache_key(project_text, candidate_ids, llm_model, top_k_llm)

            cached = _cache_load(cache_dir, key)
            if cached is not None:
                selections = cached.get("selections", []) or []
            else:
                llm_calls += 1
                try:
                    llm_json = ollama_rerank(
                        politica_candidates=candidates_for_llm,
                        project_text=project_text,
                        top_k_llm=min(top_k_llm, top_k),
                        ollama_url=ollama_url,
                        model_name=llm_model,
                        temperature=llm_temperature,
                        timeout_sec=llm_timeout_sec
                    )
                    _cache_save(cache_dir, key, llm_json)
                    selections = llm_json.get("selections", []) or []
                except Exception as e:
                    llm_failed = True
                    llm_reason = f"LLM_FAILED: {type(e).__name__}: {str(e)[:120]}"
                    selections = []

        # Construir top_k final
        used_policy = set()
        rank = 0

        # 1) LLM picks primero
        if selections:
            candidates_for_llm = candidates[: min(len(candidates), llm_candidates_cap)]
            for sel in selections:
                try:
                    cand_idx = int(sel["candidate_index"])
                    s_llm = float(sel["score"])
                    r_llm = str(sel["reason"])
                except Exception:
                    continue
                if cand_idx < 0 or cand_idx >= len(candidates_for_llm):
                    continue

                pol_j, _, s_bi = candidates_for_llm[cand_idx]
                if pol_j in used_policy:
                    continue
                used_policy.add(pol_j)
                rank += 1

                final_score = w_llm * s_llm + w_bi * float(s_bi)

                out_rows.append({
                    **base,
                    #"matched_politica_id": df_politicas.iloc[pol_j][codigo_indicador],
                    "matched_politica_text": df_politicas.iloc[pol_j][col_text_politica],
                    #"matched_indicador_producto": df_politicas.iloc[pol_j][indicador_producto],
                    #"matched_indicador_medicion": df_politicas.iloc[pol_j][medicion_indicador],
                    #"matched_indicador_unidad": df_politicas.iloc[pol_j][unidad_indicador],
                    "bi_similarity_score": float(s_bi),
                    "llm_score": float(s_llm),
                    "llm_reason": r_llm,
                    "final_score": float(final_score),
                    "rank": rank,
                    "used_llm": True,
                    "llm_failed": False,
                    "confident_by_embeddings": False,
                    "top1_bi": float(top1),
                    "margin_top1_top2": float(margin),
                    "device_used": device
                })

                if rank >= top_k:
                    break

        # 2) Completar con embeddings (si fue confiable o si LLM falló o devolvió menos)
        if rank < top_k:
            fallback = sorted(candidates, key=lambda x: x[2], reverse=True)
            for (pol_j, _, s_bi) in fallback:
                if rank >= top_k:
                    break
                if pol_j in used_policy:
                    continue
                used_policy.add(pol_j)
                rank += 1

                out_rows.append({
                    **base,
                    #"matched_politica_id": df_politicas.iloc[pol_j][codigo_indicador],
                    "matched_politica_text": df_politicas.iloc[pol_j][col_text_politica],
                    #"matched_indicador_producto": df_politicas.iloc[pol_j][indicador_producto],
                    #"matched_indicador_medicion": df_politicas.iloc[pol_j][medicion_indicador],
                    #"matched_indicador_unidad": df_politicas.iloc[pol_j][unidad_indicador],
                    "bi_similarity_score": float(s_bi),
                    "llm_score": np.nan,
                    "llm_reason": ("accepted_by_embeddings" if is_confident else ("fallback_by_embeddings" if not llm_failed else llm_reason)),
                    "final_score": float(s_bi),
                    "rank": rank,
                    "used_llm": (use_llm_rerank and (not is_confident)),
                    "llm_failed": bool(llm_failed),
                    "confident_by_embeddings": bool(is_confident),
                    "top1_bi": float(top1),
                    "margin_top1_top2": float(margin),
                    "device_used": device
                })

    df_out = pd.DataFrame(out_rows)
    df_out = df_out.sort_values([col_id_proyecto, "rank"], ascending=[True, True]).reset_index(drop=True)
    print(f"✅ Terminado. Proyectos={len(df_proyectos)} | LLM calls={llm_calls} | LLM used={llm_used} | cache_dir={cache_dir}")
    return df_out


# =========================
# Export a Excel
# =========================
def export_to_excel(df: pd.DataFrame, path_xlsx: str, project_id_col: str) -> None:
    with pd.ExcelWriter(path_xlsx, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="top_k_por_proyecto", index=False)
        top1 = df.sort_values([project_id_col, "rank"]).groupby(project_id_col, as_index=False).first()
        top1.to_excel(writer, sheet_name="top_1_por_proyecto", index=False)


In [69]:
# Si tuviste SSL antes:
# print(enable_hf_ssl_fix())

df_out = match_proyecto_to_politicas_optimizado(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica= "Indicador de Producto(MGA)",  #"Producto_enriquecido_2",
    col_text_proyecto= "Indicadores de producto PATR",  #"id_proyecto",
    col_id_proyecto="codigo_proyecto",
    #codigo_indicador="codigo_indicador",
    #indicador_producto="indicador_producto",
    #medicion_indicador="medicion_indicador",
    #unidad_indicador="unidad_indicador",

    top_k=10,

    embed_model="BAAI/bge-m3",
    batch_size=16,
    top_n_candidates=80,     # 200 suele ser suficiente y más rápido que 400
    min_bi_score=0.30,

    # Precision gating: LLM solo si embeddings no están seguros
    confident_top1_threshold=0.55,
    confident_margin_threshold=0.12,

    use_llm_rerank=False,
    llm_model= "qwen2.5:3b-instruct", #  "deepseek-r1:7b",
    llm_candidates_cap=15,
    top_k_llm=5,
    llm_timeout_sec=180,

    cache_dir="./ollama_rerank_cache"
)


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 19/19 [00:10<00:00,  1.87it/s]


[200/294] LLM calls=0, LLM used=0
✅ Terminado. Proyectos=294 | LLM calls=0 | LLM used=0 | cache_dir=./ollama_rerank_cache


In [70]:
ruta = r"C:\TrabajoAndrea\nuevo_cruce_top10_Modelo3.xlsx"
export_to_excel(df_out, ruta, project_id_col="codigo_proyecto")
print("Guardado en:", ruta)


Guardado en: C:\TrabajoAndrea\nuevo_cruce_top10_Modelo3.xlsx


# Modelo 5

In [72]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

def match_proyecto_to_politicas_embeddings_only(
    df_politicas: pd.DataFrame,
    df_proyectos: pd.DataFrame,
    col_text_politica: str,
    col_text_proyecto: str,
    col_id_proyecto: str,
    #codigo_indicador: str,
    #indicador_producto: str,
    #medicion_indicador: str,
    #unidad_indicador: str,
    top_k: int = 10,
    model_name: str = "BAAI/bge-m3",
    batch_size: int = 32,
) -> pd.DataFrame:
    """
    Top-k políticas por proyecto usando SOLO embeddings (sin LLM).
    Devuelve un df "long" con (n_proyectos * top_k) filas.
    """

    # 1) Modelo
    model = SentenceTransformer(model_name)

    # 2) Textos
    pol_texts = df_politicas[col_text_politica].fillna("").astype(str).tolist()
    proy_texts = df_proyectos[col_text_proyecto].fillna("").astype(str).tolist()

    # 3) Embeddings normalizados (coseno = dot)
    E_pol = model.encode(
        pol_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True
    )
    E_proy = model.encode(
        proy_texts, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True
    )

    # 4) Vecinos: para cada PROYECTO buscar top_k POLÍTICAS
    nn = NearestNeighbors(n_neighbors=top_k, metric="cosine", algorithm="brute")
    nn.fit(E_pol)

    distances, indices = nn.kneighbors(E_proy, return_distance=True)
    scores = 1.0 - distances  # coseno

    # 5) Construcción salida
    out_rows = []
    for i in range(len(df_proyectos)):
        base = df_proyectos.iloc[i].to_dict()
        for rank in range(top_k):
            j = int(indices[i, rank])
            sc = float(scores[i, rank])

            out_rows.append({
                **base,
                #"matched_politica_id": df_politicas.iloc[j][codigo_indicador],
                "matched_politica_text": df_politicas.iloc[j][col_text_politica],
                #"matched_indicador_producto": df_politicas.iloc[j][indicador_producto],
                #"matched_indicador_medicion": df_politicas.iloc[j][medicion_indicador],
                #"matched_indicador_unidad": df_politicas.iloc[j][unidad_indicador],
                "similarity_score": sc,
                "rank": rank + 1
            })

    return pd.DataFrame(out_rows)


In [73]:
df_out = match_proyecto_to_politicas_embeddings_only(
    df_politicas=df_politicas,
    df_proyectos=df_proyectos,
    col_text_politica= "Indicador de Producto(MGA)",  #"Producto_enriquecido_2",
    col_text_proyecto= "Indicadores de producto PATR",  #"id_proyecto",
    col_id_proyecto="codigo_proyecto",
    #codigo_indicador="codigo_indicador",
    #indicador_producto="indicador_producto",
    #medicion_indicador="medicion_indicador",
    #unidad_indicador="unidad_indicador",
    top_k=10,
    model_name="BAAI/bge-m3",
    batch_size=32
)

# ✅ Export rápido y seguro
ruta_csv = r"C:\TrabajoAndrea\nuevo_cruce_top10_modelo4.csv.csv"
df_out.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
print("CSV guardado en:", ruta_csv)
print("Filas:", len(df_out), "| Columnas:", df_out.shape[1])


Batches: 100%|█████████████████████████████████████████████████████████████████████████| 10/10 [00:08<00:00,  1.13it/s]


CSV guardado en: C:\TrabajoAndrea\nuevo_cruce_top10_modelo4.csv.csv
Filas: 2940 | Columnas: 6


In [74]:
ruta_xlsx = r"C:\TrabajoAndrea\Nuevo_cruce_top10_modelo4.xlsx"
df_out.to_excel(ruta_xlsx, index=False, engine="openpyxl")
print("Excel guardado en:", ruta_xlsx)


Excel guardado en: C:\TrabajoAndrea\Nuevo_cruce_top10_modelo4.xlsx
